# Basic Flow
Now lets try out some basic  workflow

In [1]:
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.encoders import *
from pytorch_forecasting.data.timeseries import TimeSeries
from pytorch_forecasting.metrics import MAE, SMAPE
import torch
import pandas as pd
import numpy as np
from utils import load_toydata

/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/pytorch_forecasting/models/base/_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [ ]:
# Play around with the toy dataset
num_series = 100
seq_length = 50
df = load_toydata(num_series, seq_length)
df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,-0.252836,0.109969,0,1.000000,0.810608,0
1,0,1,0.109969,0.430712,0,0.995004,0.810608,0
2,0,2,0.430712,0.603274,0,0.980067,0.810608,0
3,0,3,0.603274,0.579999,0,0.955336,0.810608,0
4,0,4,0.579999,0.849909,0,0.921061,0.810608,0


# High level API (using P layer)

Recall the process
* Create the `TimeSeries` object
* Create `configs` for model, `datamodule`, `trainer` etc.
* Create the `model_pkg` object
* perform `pkg.fit` and `pkg.predict`.


## Thoughts
- Why is the tutorial repeating stuff in exmaples

In [10]:
from pytorch_forecasting.data.timeseries import TimeSeries


dataset = TimeSeries(
    data=df,
    time="time_idx",
    target="y",
    num=["static_feature"],
    cat=["category","static_feature_cat"],
)



/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/pytorch_forecasting/data/timeseries/_timeseries_v2.py:104: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


Create the configs

In [26]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from pytorch_forecasting.data.encoders import TorchNormalizer
data_module_cfg = dict(
    max_encoder_lenght=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": MinMaxScaler(),
    },
    target_normalizer=TorchNormalizer(),
)


Choose the models from [here](https://pytorch-forecasting.readthedocs.io/en/stable/model_list.html#v2-models). 

In [28]:
model_cfg = dict(
    optimizer="sgd",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
)


See at the [doc](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.trainer.trainer.Trainer.html#lightning.pytorch.trainer.trainer.Trainer) of trainer to see to play aorund with the inputs of `Trainer`.

In [29]:
trainer_cfg = dict(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

Create the `pkg` class

In [30]:
from pytorch_forecasting.models.tide._tide_dsipts._tide_v2 import TIDE

In [32]:
model_pkg = TIDE(
    model_cfg=model_cfg,
    trainer_cfg=trainer_cfg,
    datamodule_cfg=data_module_cfg,
)

TypeError: TIDE.__init__() missing 7 required positional arguments: 'metadata', 'loss', 'hidden_size', 'd_model', 'n_add_enc', 'n_add_dec', and 'dropout_rate'

Try `fit`, `predict`.

In [ ]:
# TODO: write fit, predict scripts using the model_pkg object
model_pkg.fit(...)
preds = model_pkg.predict(...)

In [ ]:
print(preds)

## 3 stage pipeline

Recall the steps
1. Create `TimeSeries` Dataset object
2. Create DataModule object
3. Initialize, Train & Run Inference with the Model


Choose the models from [here](https://pytorch-forecasting.readthedocs.io/en/stable/model_list.html#v2-models). 

You can see the compatible Datamodules in the table along side the model names to see which datamoudle can be used with which model!

`TimeSeries` class

In [18]:
dataset = TimeSeries(
    data=df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)


The datamodule

In [19]:
data_module = EncoderDecoderTimeSeriesDataModule(
    time_series_dataset=dataset,
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:129: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


Initialise the model

Choose the models from [here](https://pytorch-forecasting.readthedocs.io/en/stable/model_list.html#v2-models). 

In [ ]:
from pytorch_forecasting.models.samformer._samformer_v2 import Samformer

In [21]:
model = Samformer(
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    use_revin=True,
    metadata=data_module.metadata
)

/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/pytorch_forecasting/models/base/_base_model_v2.py:85: UserWarning: The Model 'Samformer' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing and as a basic skeleton, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


In [22]:
from lightning.pytorch import Trainer

In [23]:
trainer = Trainer(
    max_epochs=5,
    accelerator="auto",
    enable_progress_bar=True
)

trainer.fit(model, data_module)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litm

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (42) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 4: 100%|██████████| 42/42 [00:01<00:00, 23.79it/s, v_num=2, train_loss_step=24.60, val_loss=9.710, val_MAE=9.710, val_SMAPE=1.220, train_loss_epoch=19.40, train_MAE=19.40, train_SMAPE=1.060]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 42/42 [00:01<00:00, 23.73it/s, v_num=2, train_loss_step=24.60, val_loss=9.710, val_MAE=9.710, val_SMAPE=1.220, train_loss_epoch=19.40, train_MAE=19.40, train_SMAPE=1.060]


In [24]:
preds = trainer.predict(model, data_module)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/siddharth/work/sktime_oss/pytorch_forecasting_user_session/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 60/60 [00:01<00:00, 35.85it/s]


In [25]:
print(preds)

[{'prediction': tensor([[[-0.7441]],

        [[ 0.8629]],

        [[ 3.3040]],

        [[ 4.9477]],

        [[ 6.3637]],

        [[ 7.7176]],

        [[ 8.5714]],

        [[ 9.2498]],

        [[ 9.9976]],

        [[ 9.8799]],

        [[ 9.3374]],

        [[ 8.5412]],

        [[ 7.3346]],

        [[ 5.9947]],

        [[ 4.0749]],

        [[ 2.3194]],

        [[ 0.5538]],

        [[-1.6303]],

        [[-3.3838]],

        [[-0.7480]],

        [[ 0.8788]],

        [[ 3.1838]],

        [[ 5.0721]],

        [[ 6.5731]],

        [[ 7.7781]],

        [[ 8.7713]],

        [[ 9.4789]],

        [[ 9.8958]],

        [[ 9.9370]],

        [[ 9.4747]],

        [[ 8.5636]],

        [[ 7.3107]]])}, {'prediction': tensor([[[ 5.9352]],

        [[ 4.1110]],

        [[ 2.3210]],

        [[ 0.5757]],

        [[-1.4204]],

        [[-3.2317]],

        [[-0.7586]],

        [[ 0.8835]],

        [[ 3.2128]],

        [[ 5.0811]],

        [[ 6.5530]],

        [[ 7.7843]],


## Play around by trying different models, etc. 